# Промпт #20 — Комбинация техник edge case (сарказм)

**Техника:** System prompt + Few-shot + CoT + JSON вывод  
**Задача:** Устоит ли комбинация на саркастичном отзыве?  
**Сложность:** ⭐⭐⭐⭐☆

In [1]:
import sys
sys.path.append('..')
from config import get_completion
import json

In [2]:
system = """Ты эксперт по анализу отзывов клиентов. 
Всегда думай пошагово перед ответом.
Отвечай только валидным JSON без markdown-фенсов."""

prompt = """Проанализируй отзыв и верни JSON с полями:
- тональность: ПОЗИТИВНЫЙ / НЕГАТИВНЫЙ / НЕЙТРАЛЬНЫЙ
- причина: одно предложение почему
- рекомендация: что сделать бизнесу

Примеры:
<examples>
<example>
Отзыв: "Доставка быстрая, товар exactly как на фото!"
Ответ: {"тональность": "ПОЗИТИВНЫЙ", "причина": "Клиент доволен скоростью и качеством", "рекомендация": "Поддерживать текущий уровень сервиса"}
</example>
<example>
Отзыв: "Ждал месяц, привезли не то что заказывал"
Ответ: {"тональность": "НЕГАТИВНЫЙ", "причина": "Долгая доставка и ошибка в заказе", "рекомендация": "Улучшить логистику и контроль сборки заказов"}
</example>
</examples>

Отзыв: "Ну просто замечательно! Заказал 2 недели назад, курьер приехал когда меня не было, перезвонить не удосужился. Теперь жду возврата денег уже 10 дней. Просто восхитительный сервис!" """

response = get_completion(prompt, system_prompt=system, temperature=0.1)
print("=== ОТВЕТ МОДЕЛИ ===")
print(response)

try:
    clean = response.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    parsed = json.loads(clean)
    print("\n✅ Валидный JSON")
    print(f"Тональность: {parsed['тональность']}")
    print(f"Причина: {parsed['причина']}")
    print(f"Рекомендация: {parsed['рекомендация']}")
except json.JSONDecodeError:
    print("\n❌ Невалидный JSON")

=== ОТВЕТ МОДЕЛИ ===
{"тональность": "НЕГАТИВНЫЙ", "причина": "Клиент недоволен качеством обслуживания и возвратом денег", "рекомендация": "Улучшить коммуникацию с клиентами и процесс возврата денег"}

✅ Валидный JSON
Тональность: НЕГАТИВНЫЙ
Причина: Клиент недоволен качеством обслуживания и возвратом денег
Рекомендация: Улучшить коммуникацию с клиентами и процесс возврата денег


## Оценка: 5/5

## Инсайт
Комбинация устояла на сарказме — НЕГАТИВНЫЙ, валидный JSON, без фенсов.

Модель не повелась на слова "замечательно" и "восхитительный" — поняла иронию
через контекст (курьер не перезвонил, ждёт возврат 10 дней).

Сравнение с #04 (сарказм без комбинации): там тоже сработало, но здесь
результат структурированный и сразу готов для использования в продукте —
можно передавать в базу данных или дашборд без дополнительной обработки.

Итог блока #15-20:
- System prompt контролирует поведение, user prompt работает внутри рамок (#15-16)
- Температура влияет на структуру больше чем на креативность (#17-18)
- Комбинация техник закрывает слабости каждой в отдельности (#19-20)